# Cross-Dataset Evaluation / 跨数据集评估

`04_leakage_audit` 的结论是：ISOT 里的真新闻全部来自路透社，模型靠"路透社文风"就能拿到高分，清洗也无法消除这一点。本 notebook 回答下一个问题：**在 ISOT 上训练的模型，换一批来源后还剩多少能力？**

- **Part A（本部分，步骤 1a）**：选择外部数据集，下载、校验，并按和 ISOT 完全相同的方式处理
- **Part B（步骤 1c）**：跨数据集实验

`04_leakage_audit` showed that ISOT's real news is all Reuters, so a model can score highly on "Reuters style" — something cleaning cannot remove. This notebook asks: **how much ability does an ISOT-trained model keep on data from different sources?** Part A selects, downloads, verifies and processes the external data exactly like ISOT; Part B runs the experiments.

## 1. Setup / 环境

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.utils import set_seed
from src.data import load_dataset, to_model_text
from src.external import (load_welfake_raw, load_welfake_external, load_welfake_report,
                          load_liar, verify_welfake_label_direction, _fingerprints, _key_set,
                          _english_share, MIN_ENGLISH_SHARE, WELFAKE_MD5, LIAR_MD5, EXTERNAL_DIR, _md5)
from src.data import load_raw
from src import probes

set_seed(42)
pd.set_option('display.max_colwidth', 100)

/Users/sl1425/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Choosing the external dataset / 选择外部数据集

理想的外部数据集应满足三个条件：**全文新闻**（和 ISOT 格式一致）、**真新闻不只来自路透社**、**可以合法使用**。检查过的三个候选：

| 数据集 | 内容 | 许可 | 结论 |
|---|---|---|---|
| FakeNewsNet（Hugging Face 版） | **只有标题和链接**，没有正文；大部分是 GossipCop 娱乐新闻 | CC | ❌ 不适用：没有正文，完整版需要逐条抓取，很多链接已失效 |
| LIAR（Wang, 2017） | 12,836 条 PolitiFact 逐条核查的短陈述（中位数 17 个词） | 仅限研究用途 | ⚠️ 次要：标签直接反映真假，但只有一句话，和长文章格式差异太大 |
| **WELFake**（Verma et al., 2021） | 72,134 篇全文新闻，合并了 Kaggle、McIntire、Reuters、BuzzFeed 四个数据集 | **CC-BY-4.0** | ✅ 主要：**去掉其中的 ISOT 部分后**，真新闻来自《纽约时报》、Breitbart 等，不是路透社 |

An external set should be **full-text**, have **real news beyond Reuters**, and be **legally usable**. FakeNewsNet (HF) has titles only; LIAR is one-sentence statements (secondary); **WELFake minus its ISOT portion** is the primary choice (CC-BY-4.0).

> **重要说明 / Caveat**：WELFake 的标签同样是 **按来源网站** 打的（例如 Breitbart 整体被标为"真"），不是逐篇核查。所以它检验的是"路透社文风学到的东西能否迁移到其他来源"，而不是"模型能否判断真假"。后者需要 LIAR 这类逐条核查的数据，或者步骤 5 的证据核查方法。
> WELFake labels are also **per source site** (Breitbart is "real" wholesale), so it tests whether what was learned from Reuters style transfers to other sources — not truthfulness itself.

数据来源 / Sources:
- WELFake: Verma, Agrawal & Prodan (2021), *IEEE Transactions on Computational Social Systems*. Zenodo, [doi:10.5281/zenodo.4561253](https://doi.org/10.5281/zenodo.4561253), CC-BY-4.0
- LIAR: Wang (2017), ["Liar, Liar Pants on Fire"](https://aclanthology.org/P17-2067/), ACL 2017. Research use only.

## 3. Download & checksum / 下载与校验

文件下载到 `data/external/`（不提交到 git）。WELFake 的 MD5 来自 Zenodo 官方 API；LIAR 官方没有公布校验值，记录的是第一次下载时的 MD5，保证以后拿到的是同一份文件。

Files go to `data/external/` (not in git). The WELFake MD5 is Zenodo's official value; LIAR has none, so the MD5 from the first download is pinned.

In [2]:
for name, expected in [('WELFake_Dataset.csv', WELFAKE_MD5), ('liar_dataset.zip', LIAR_MD5)]:
    path = EXTERNAL_DIR / name
    actual = _md5(path)
    print(f'{name:22s} {path.stat().st_size / 1e6:7.1f} MB  md5 {actual}  '
          f'{"OK" if actual == expected else "MISMATCH"}')

WELFake_Dataset.csv      245.1 MB  md5 73c9675a4b3d09f86a6933d0b8d7d908  OK
liar_dataset.zip           1.0 MB  md5 437541ca9957e993634f9c1e71a26748  OK


## 4. Label direction check / 标签方向检查

Zenodo 页面写的是 "0 = fake and 1 = real"。WELFake 里包含 ISOT 的原文，可以用它们来验证：**ISOT 真新闻在 WELFake 里被标成了什么？**

`load_welfake_raw()` 已经按下面的验证结果把标签转换成本项目的约定（1 = 真，0 = 假）。转换正确时，下表应该几乎只有对角线上有数字。

The Zenodo page says "0 = fake and 1 = real". Since WELFake contains ISOT articles, we can check what ISOT's real news is labelled as. `load_welfake_raw()` converts to this project's convention (1 = real); after a correct conversion the table below is diagonal.

In [3]:
welfake_raw = load_welfake_raw()
isot_raw = load_raw()
verify_welfake_label_direction(welfake_raw, isot_raw)

WELFake label (converted),0,1
ISOT label,,
0,22641,1
1,0,21416


转换后两边几乎完全一致（44,058 篇里只有 1 篇不同）。这说明 **原始 CSV 实际上是 1 = 假、0 = 真**，正好和 Zenodo 页面的说明相反；论文中"37,106 篇假新闻"的数量也和原始 `label == 1` 的数量一致。如果照页面说明直接使用，所有跨数据集结果都会完全反过来，而且很难察觉。`verify_welfake_label_direction()` 每次构建时都会重新检查，一致率低于 99.9% 就会报错。

After conversion the labels agree on all but 1 of 44,058 articles: the **raw CSV is actually 1 = fake, 0 = real**, the opposite of the Zenodo page (the paper's "37,106 fake" equals the raw count of `label == 1`). Used as documented, every cross-dataset result would be silently inverted. The check re-runs on every build and fails below 99.9% agreement.

## 5. Building "WELFake minus ISOT" / 构建外部测试集

处理方式和 ISOT 完全相同（同一套 `clean_title` / `clean_text` / `to_model_text`），另外有五项外部数据专用的步骤：
1. 删除标题末尾的出版方后缀，例如 " - The New York Times"（真新闻 63%、假新闻 2.6%，和 ISOT 的 "[VIDEO]" 一样是来源标签）
2. 删除结尾的推广语和作者简介（例如 Breitbart 的 "Follow him on Twitter… email him at…"、《纽约时报》简报页脚、假新闻网站的 "please like and share"），以及 "Breitbart" 这个来源名称。判断方法：命中推广语规则，或者同一句话逐字出现在 10 篇以上文章的结尾。这些模式在 ISOT 中各项都 ≤ 1.4%，在 WELFake 中却有 146 种固定结尾、覆盖 2,865 篇文章
3. 删除非英文文章
4. 删除和 ISOT **任何部分**（train + val + test）有关联的文章：先用原文比对，再比较五种指纹
5. 和 ISOT 一样做近似重复分组和按组分层划分（64/16/20），供 Part B 的参照实验使用

Processing is identical to ISOT, plus five external-only steps: strip publisher suffixes; strip trailing promos / author bios (rule match, or a sentence repeated verbatim at the end of 10+ articles) and the source name "Breitbart"; drop non-English articles; drop anything linked to ANY part of ISOT; and group + split 64/16/20 like ISOT for Part B.

In [4]:
welfake = load_welfake_external()
report = load_welfake_report()

steps = pd.DataFrame({k: report[k] for k in [
    'removed_isot_copies_raw_match', 'removed_short_body', 'removed_non_english',
    'removed_exact_duplicates', 'removed_isot_overlap_fingerprint', 'final_by_label']}).T
steps.index = ['ISOT 副本（原文比对）/ ISOT copies (raw match)',
               '正文 < 20 词 / body < 20 words',
               '非英文 / non-English',
               '完全重复 / exact duplicates',
               'ISOT 近似副本（指纹）/ ISOT near-copies (fingerprints)',
               '最终 / final']
print(f"Raw WELFake rows: {report['raw_total']:,}   conflicting-label rows removed: "
      f"{report['removed_conflicting_label_rows']}")
steps

Raw WELFake rows: 72,134   conflicting-label rows removed: 0


,fake,real
ISOT 副本（原文比对）/ ISOT copies (raw match),22641,21417
正文 < 20 词 / body < 20 words,1462,188
非英文 / non-English,447,2
完全重复 / exact duplicates,2573,16
ISOT 近似副本（指纹）/ ISOT near-copies (fingerprints),49,18
最终 / final,9934,13387


几点说明 / Notes:
- **ISOT 副本占了 WELFake 的六成以上**（44,058 / 72,134）。如果不去掉，跨数据集测试其实就是在 ISOT 上测试。
- **原文比对之后，指纹又找出了 67 篇** 和 ISOT 相关的近似副本（原文有细微差异、或者换了标题 / 开头）。只做原文比对是不够的。
- **非英文文章几乎全是假新闻**（447 篇假、2 篇真），保留的话"不是英文"本身就会成为一个假新闻信号。
- 最终 **23,321 篇**，真新闻占 57.4%。

- **ISOT copies make up over 60% of WELFake**; without removing them, "cross-dataset" would really be testing on ISOT.
- **Fingerprints catch 67 more** near-copies after the raw match — a raw match alone is not enough.
- **Non-English articles are almost all fake**; kept, "not English" would itself signal fake.
- Final: **23,321 articles**, 57.4% real.

## 6. Verification / 验证

三项独立检查 / Three independent checks:
1. 和清洗后的 ISOT（全部 37,982 篇）比较五种指纹，**必须全部为 0**
2. 不应残留非英文文章
3. 残留的来源格式痕迹，和 ISOT 清洗后的情况并排比较

In [5]:
isot = load_dataset()
fp_isot = _fingerprints(isot['title'], isot['text'], isot['model_text'])
fp_ext = _fingerprints(welfake['title'], welfake['text'], welfake['model_text'])
overlap = {name: int(fp_ext[name].isin(_key_set(fp_isot[name])).sum()) for name in fp_ext}
print('1) external articles sharing a fingerprint with ISOT:', overlap)
print('2) non-English articles left:',
      int((_english_share(welfake['model_text']) < MIN_ENGLISH_SHARE).sum()))

Loaded cached dataset: 37,982 rows (clean_dataset.csv)


1) external articles sharing a fingerprint with ISOT: {'exact': 0, 'title': 0, 'lead': 0, 'tail': 0, 'middle': 0}


2) non-English articles left: 0


In [6]:
pd.concat({'ISOT (cleaned)': probes.artefact_rates(isot),
           'WELFake-external': probes.artefact_rates(welfake)}, axis=1)

ISOT (cleaned)        WELFake-external       
                                      fake % real %           fake % real %
Reuters 字样 / "Reuters"                   0.0    0.1              0.4    0.1
开头署名 / dateline "(Reuters) -"            0.0    0.0              0.0    0.0
空括号 / empty "()"                         0.0    0.0              0.0    0.0
记者署名 / "(Reporting by"                   0.0    0.0              0.0    0.0
图片来源 / "Featured image"                  0.0    0.0              0.0    0.0
Getty Images                             0.1    0.0              0.4    0.2
转载来源 / "Via:" / "Read more"              0.0    0.0              0.0    0.0
推文链接 / pic.twitter.com                   0.0    0.0              0.0    0.0
URL                                      0.0    0.0              0.0    0.1
@用户名 / @handle                           0.0    0.0              0.0    0.0
弯引号 / curly quotes                       0.0    0.0              0.0    0.0
不换行空格 / \xa0                             0.0    0.0              0.0    0.0
句号后缺空格 / "word.Word"                     0.0    0.0              0.0    0.0
标题方括号 / title "[...]"                    0.0    0.0              0.0    0.0
标题 VIDEO / WATCH                         3.0    0.3              1.2    0.7
标题感叹号 / title "!"                        8.4    0.1              4.9    0.4
文风 / "said"                             57.7   93.7             44.8   82.5
文风 / "U.S."                             16.6   60.2             19.5    7.8

格式类痕迹在两个数据集中都接近 0，所以跨数据集测试不会被这些痕迹干扰。

值得注意的是最后两行的 **文风** 特征：
- **"said"** 在两个数据集中都是真新闻多得多（ISOT 94% vs 58%；WELFake 83% vs 45%），因为《纽约时报》和路透社一样，都是大量引用消息来源的新闻写法。所以 ISOT 学到的 "said" 线索，在 WELFake 上 *仍然有用*。
- **"U.S."** 则正好 **反过来**：在 ISOT 里是真新闻多（60% vs 17%），在 WELFake 里是假新闻多（20% vs 8%）。不过 `model_text` 会把 "U.S." 和 "US" 合并，所以这个具体差异不会直接进入模型。

这说明文风线索换了数据集后，**有的能迁移，有的会失效甚至反向**。所以跨数据集的结果不会是简单的"全对"或"全错"，Part B 需要结合特征权重一起解读。

Formatting artefacts are ~0 in both sets. Style is mixed: **"said"** favours real news in both datasets (NYT, like Reuters, is attribution-heavy), so that cue *still helps* on WELFake; **"U.S."** flips direction between the two. Some style cues transfer and some reverse, so Part B results must be read together with feature weights.

## 7. Where does the external real news come from? / 外部真新闻的来源

清洗时出版方后缀已经被删掉，这里回到原始标题，统计最终保留的真新闻来自哪些出版方。

Publisher suffixes were stripped during cleaning, so we count them on the raw titles of the articles that were kept.

In [7]:
raw_titles = welfake_raw.set_index('doc_id').loc[welfake['doc_id'], 'title']
publisher = raw_titles.str.extract(r' - (The New York Times|Breitbart)\s*$')[0].fillna('no suffix (other / unknown)')
pd.crosstab(publisher.values, welfake['label'].map({0: 'fake', 1: 'real'}).values,
            margins=True, margins_name='total')

col_0,fake,real,total
row_0,,,
Breitbart,1,2331,2332
The New York Times,0,6219,6219
no suffix (other / unknown),9933,4837,14770
total,9934,13387,23321


外部真新闻里，46%（6,219 篇）来自《纽约时报》，17%（2,331 篇）来自 Breitbart，其余 36% 没有出版方后缀（来自 McIntire 和 BuzzFeed 数据集中的多家媒体）。**没有一篇来自路透社**，这正是我们需要的：ISOT 模型如果只学会了"路透社文风"，在这里就会失效。

46% of the external real news is NYT (6,219) and 17% Breitbart (2,331); the remaining 36% carries no suffix (multiple outlets from the McIntire and BuzzFeed subsets). **None is Reuters** — exactly what is needed to test whether an ISOT model learned more than Reuters style.

## 8. LIAR overview / LIAR 概况

LIAR 作为 **次要** 外部测试集：每条都是 PolitiFact 编辑逐条核查过的陈述，标签直接反映真假，但只有一句话。6 级标签按文献中最常见的方式转为二分类：true / mostly-true / half-true → 真，barely-true / false / pants-fire → 假。`label6` 保留了原始 6 级标签，以后可以尝试其他划分方式（例如把 half-true 去掉）。

LIAR is the **secondary** external set: individually fact-checked, but one sentence each. The 6 labels are binarised the most common way (true / mostly-true / half-true → real); `label6` keeps the original labels for other mappings later.

In [8]:
liar = load_liar()
print(liar.groupby('split')['label'].agg(n='size', real_share='mean').round(3))
print()
print('Original 6-way labels:', liar['label6'].value_counts().to_dict())
print('Median length (words):  LIAR', int(liar['text'].str.split().str.len().median()),
      '| WELFake-external', int(welfake['text'].str.split().str.len().median()),
      '| ISOT', int(isot['text'].str.split().str.len().median()))

           n  real_share
split                   
test    1283       0.567
train  10269       0.562
val     1284       0.520

Original 6-way labels: {'half-true': 2638, 'false': 2511, 'mostly-true': 2466, 'barely-true': 2108, 'true': 2063, 'pants-fire': 1050}


Median length (words):  LIAR 17 | WELFake-external 609 | ISOT 368


## 9. Part A summary / A 部分小结

| | WELFake-external（主要） | LIAR（次要） |
|---|---|---|
| 规模 | 23,321 篇（真 57.4%），按组划分 14,855 / 3,775 / 4,691 | 12,836 条（官方划分 10,269 / 1,284 / 1,283） |
| 格式 | 全文新闻，和 ISOT 一致 | 一句话陈述（中位数 17 个词） |
| 真新闻来源 | 《纽约时报》、Breitbart 等，**没有路透社** | PolitiFact 核查的各类陈述 |
| 标签依据 | 按来源网站 | **逐条核查** |
| 和 ISOT 的重叠 | 0（五种指纹） | 不同类型的数据，不存在全文重叠 |
| 许可 | CC-BY-4.0 | 仅限研究用途 |

**处理过程中发现并解决的问题 / Issues found and fixed:**
1. WELFake 的标签方向和官方说明 **相反**，已转换，并在每次构建时自动验证
2. WELFake 六成以上是 ISOT 原文，已全部删除，包括 67 篇近似副本
3. 外部真新闻标题带出版方后缀（63%），属于来源泄漏，已删除
4. 结尾的推广语、作者简介和 "Breitbart" 来源名（146 种固定结尾、2,865 篇），已删除；为防止误删正文，单句超过 80 词或累计超过 150 词就停止
5. 非英文文章几乎全是假新闻（447 / 449），已删除
6. pandas `isin()` 会把 NaN 和 NaN 视为相等，可能把大量文章误判为重叠，已修复并加了单元测试

---
# Part B — Cross-dataset experiments / 跨数据集实验

## 10. Design / 实验设计

| 实验 | 训练 | 选择超参数 | 评估 | 回答什么 |
|---|---|---|---|---|
| **E1** | ISOT train | ISOT val | **WELFake test** | 在 ISOT 上学到的东西，换一批来源后还剩多少？ |
| **E2**（参照） | WELFake train | WELFake val | **WELFake test** | 同一个模型在这个外部数据集 *内部* 能做到多好？ |

四条规则，避免常见的错误 / Four rules against common mistakes:
1. **两个实验在同一批测试文章上评估**（WELFake test，4,691 篇），差距只来自训练数据。
2. **超参数只在训练数据所在数据集的验证集上选择**，绝不用目标测试集调参。ISOT 的 test 集在本 notebook 中完全没有使用。
3. **必须和多数类基线比较**：WELFake test 中 56.1% 是真新闻，所以"全部猜真"就有 56.1% 的准确率。
4. **同时报告 AUC 和 balanced accuracy**：换数据集后，模型打分可能整体偏移，0.5 阈值下的准确率会产生误导；AUC 不依赖阈值。另外用 bootstrap 给出 95% 置信区间。

模型是 TF-IDF + 逻辑回归（`src/baselines.py`），设置和 `04_leakage_audit` 的探针相同，只有正则化强度 C 需要选择。

E1 trains on ISOT, E2 (reference) on WELFake; both are scored on the **same** WELFake test articles. C is tuned only on the training dataset's own validation split (ISOT test is never used here). Accuracy is read against the 56.1% majority baseline, and AUC / balanced accuracy with bootstrap 95% CIs are reported because scores can shift across datasets.

## 11. Data for the two experiments / 两个实验的数据

In [9]:
from src import baselines as B
from src.data import get_splits

isot_train, isot_val, _ = get_splits(isot)              # 第三个是 ISOT test，这里不使用 / ISOT test — unused
wel_train, wel_val, wel_test = get_splits(welfake)

pd.DataFrame({name: {'n': len(d), 'real %': round(100 * d['label'].mean(), 1)}
              for name, d in [('ISOT train', isot_train), ('ISOT val', isot_val),
                              ('WELFake train', wel_train), ('WELFake val', wel_val),
                              ('WELFake test', wel_test)]}).T

,n,real %
ISOT train,24278.0,55.1
ISOT val,6074.0,55.0
WELFake train,14855.0,57.7
WELFake val,3775.0,57.9
WELFake test,4691.0,56.1


In [10]:
print('WELFake near-duplicate groups spanning more than one split:', probes.groups_cross_splits(welfake))
probes.split_overlap(welfake)

WELFake near-duplicate groups spanning more than one split: 0


,val,test
完全相同 / exact model_text,0,0
标题相同 / same title,0,0
开头 25 词相同 / same lead,0,0
结尾 25 词相同 / same tail,2,0
中间 25 词相同 / same middle (independent check),2,3


WELFake 的划分和 ISOT 用的是同一种方法：近似重复分组 + 按组分层划分，所以 E2 的参照分数不会被重复文章抬高。"结尾相同"一行只剩极少数（被超过 25 篇文章共用的结尾不参与分组，见 `src/data.py` 的 `NEAR_DUP_MAX_TAIL_GROUP`）。

WELFake is split the same way as ISOT (near-duplicate groups, grouped stratified split), so duplicates cannot inflate E2.

## 12. Choosing C / 选择正则化强度 C

每个实验只在 **自己训练数据的验证集** 上选 C，按 AUC 挑最好的。C 越大，正则化越弱，模型越贴合训练数据。并列时选较小的 C（正则化更强）。

C is chosen on each training dataset's own validation split, by AUC (ties → smaller C).

In [11]:
C_isot, table_isot = B.select_C(isot_train, isot_val)
C_wel, table_wel = B.select_C(wel_train, wel_val)
print(f'E1 (ISOT val)    -> C = {C_isot}')
print(f'E2 (WELFake val) -> C = {C_wel}')
pd.concat({'E1: ISOT val': table_isot, 'E2: WELFake val': table_wel}, axis=1)

E1 (ISOT val)    -> C = 64.0
E2 (WELFake val) -> C = 64.0


E1: ISOT val         E2: WELFake val        
       val accuracy % val AUC  val accuracy % val AUC
C                                                    
0.25          96.8225  0.9942         91.3642  0.9747
1.00          97.6128  0.9958         93.7219  0.9843
4.00          98.3042  0.9976         94.5166  0.9882
16.00         98.5347  0.9978         94.9669  0.9897
64.00         98.5512  0.9980         95.1523  0.9900
256.00        98.5677  0.9980         94.8344  0.9889

两个数据集都选中 C = 64，而且都不在候选范围的边界上（WELFake 到 256 时下降；ISOT 在 64 和 256 时并列，按规则选较小的 64），说明搜索范围足够。

Both pick C = 64, inside the grid (WELFake drops at 256; ISOT ties at 64 / 256 and the smaller is kept).

## 13. Main results / 主要结果

In [12]:
vec_isot, clf_isot = B.fit_tfidf_lr(isot_train, C_isot)
vec_wel, clf_wel = B.fit_tfidf_lr(wel_train, C_wel)

prob_e1 = B.predict_proba_real(vec_isot, clf_isot, wel_test)      # E1: ISOT 模型 → WELFake test
prob_e2 = B.predict_proba_real(vec_wel, clf_wel, wel_test)        # E2: WELFake 模型 → WELFake test
prob_isot_val = B.predict_proba_real(vec_isot, clf_isot, isot_val)

results = pd.DataFrame({
    'ISOT model → ISOT val (in-domain, for reference)': B.evaluate(isot_val['label'], prob_isot_val),
    'E1: ISOT model → WELFake test': B.evaluate(wel_test['label'], prob_e1),
    'E2: WELFake model → WELFake test': B.evaluate(wel_test['label'], prob_e2),
}).T
results

,n,real %,majority baseline %,accuracy %,balanced accuracy %,AUC,F1 fake,F1 real,predicted real %
"ISOT model → ISOT val (in-domain, for reference)",6074.0,55.0,55.0,98.6,98.5,0.998,0.984,0.987,55.3
E1: ISOT model → WELFake test,4691.0,56.1,56.1,56.7,60.4,0.714,0.648,0.438,20.9
E2: WELFake model → WELFake test,4691.0,56.1,56.1,95.0,94.8,0.990,0.942,0.956,57.1


In [13]:
ci = pd.DataFrame({'E1: ISOT → WELFake test': B.bootstrap_ci(wel_test['label'], prob_e1),
                   'E2: WELFake → WELFake test': B.bootstrap_ci(wel_test['label'], prob_e2)}).T
ci

,balanced accuracy 95% CI,AUC 95% CI
E1: ISOT → WELFake test,"(59.4, 61.5)","(0.7, 0.73)"
E2: WELFake → WELFake test,"(94.2, 95.4)","(0.988, 0.992)"


### 怎么读这张表 / Reading the table

- **ISOT 模型在 ISOT 内部：98.6%。** 略高于 `04_leakage_audit` 探针的 98.3%，因为这里的 C 经过选择（64，审计中固定为 4）。
- **同一个模型换到 WELFake：准确率 56.7%，而多数类基线是 56.1%。** 也就是说，它几乎不比"全部猜真"更好。
- **balanced accuracy 60.4%**（95% 置信区间 59.4–61.5），**AUC 0.714**（0.70–0.73）。AUC 明显高于 0.5，说明模型 *还保留了一些* 排序能力，即平均来说，它给真新闻的分数仍然比假新闻高一点。但远远达不到在 ISOT 上的水平。
- **它只把 20.9% 的文章判为真，而实际有 56.1% 是真新闻。** 换了来源之后，大部分真新闻被判成了假新闻。
- **E2 参照：同样的模型在 WELFake 内部训练，可以达到 95.0%，AUC 0.990。** 所以 WELFake 本身并不难，问题出在 *从 ISOT 迁移过来*。
- 两个实验的置信区间完全不重叠（59.4–61.5 vs 94.2–95.4），所以这个差距不是测试集抽样带来的偶然。

- **In-domain 98.6%**, but on WELFake **56.7% accuracy against a 56.1% majority baseline** — barely better than always guessing "real".
- **Balanced accuracy 60.4% (59.4–61.5), AUC 0.714 (0.70–0.73)**: some ranking ability survives, far from in-domain.
- It predicts only **20.9% real** while **56.1%** are real: most real news from other sources is called fake.
- **E2 reaches 95.0% / AUC 0.990** on the same test articles, so WELFake itself is not hard; the problem is transfer from ISOT. The CIs do not overlap.

## 14. Where does E1 go wrong? / E1 错在哪里

In [14]:
pd.concat({'E1: ISOT model': B.confusion_table(wel_test['label'], prob_e1),
           'E2: WELFake model': B.confusion_table(wel_test['label'], prob_e2)}, axis=1)

E1: ISOT model                E2: WELFake model               
          predicted fake predicted real    predicted fake predicted real
true fake           1871            189              1919            141
true real           1840            791                95           2536

In [15]:
# 按真新闻的出版方分组，看 ISOT 模型把多少比例判为"真"
# Share predicted REAL, by publisher of the real articles (from the raw title suffix)
raw_titles = welfake_raw.set_index('doc_id').loc[wel_test['doc_id'], 'title']
publisher = raw_titles.str.extract(r' - (The New York Times|Breitbart)\s*$')[0].fillna('other outlets').values
group = np.where(wel_test['label'] == 0, 'FAKE articles (all)', 'REAL: ' + publisher)

by_source = (pd.DataFrame({'group': group, 'E1': prob_e1 > 0.5, 'E2': prob_e2 > 0.5})
               .groupby('group')
               .agg(n=('E1', 'size'),
                    **{'E1 predicted real %': ('E1', lambda x: round(100 * x.mean(), 1)),
                       'E2 predicted real %': ('E2', lambda x: round(100 * x.mean(), 1))}))
by_source

,n,E1 predicted real %,E2 predicted real %
group,,,
FAKE articles (all),2060,9.2,6.8
REAL: Breitbart,440,10.5,93.4
REAL: The New York Times,1251,43.2,99.3
REAL: other outlets,940,21.8,93.9


- **假新闻：两个模型都能识别**（ISOT 模型只把 9.2% 判为真）。
- **真新闻：ISOT 模型几乎全军覆没。** 《纽约时报》只有 43.2% 被判为真，其他媒体 21.8%，Breitbart 只有 10.5%。WELFake 模型对这三组都在 93% 以上。

换句话说，ISOT 模型的实际行为更接近 **"像不像路透社？不像就判假"**。

**关于 Breitbart 的说明**：WELFake 把 Breitbart 整体标为"真"，这个标签本身有争议，所以 ISOT 模型在 Breitbart 上的"错误"不一定是真正的错误。但 **《纽约时报》是没有争议的主流媒体**，ISOT 模型仍然把其中 56.8% 判为假新闻。只看这一组，结论也成立。

Fake articles are caught by both models, but the ISOT model calls most *real* news fake: only 43.2% of NYT, 21.8% of other outlets and 10.5% of Breitbart are predicted real (E2: 93%+ for all). It behaves like "Reuters-like or fake". Breitbart's "real" label is debatable, but NYT is not — and 56.8% of NYT articles are still called fake.

## 15. Why? Comparing what the two models learned / 为什么？比较两个模型学到了什么

逻辑回归给每个词或词组一个权重：正数推向"真"，负数推向"假"。比较两个模型对 **同一批词** 的权重，就能看出 ISOT 学到的线索在 WELFake 上还成不成立。

Each feature gets a weight (positive → real, negative → fake). Comparing the two models' weights on shared vocabulary shows which ISOT cues still hold on WELFake.

In [16]:
coef_isot = B.coefficient_table(vec_isot, clf_isot)
coef_wel = B.coefficient_table(vec_wel, clf_wel)
shared = coef_isot.index.intersection(coef_wel.index)
rank_corr = coef_isot[shared].corr(coef_wel[shared], method='spearman')
print(f'Features in both vocabularies: {len(shared):,} of 50,000')
print(f'Spearman correlation of their weights: {rank_corr:.3f}   (1 = same ranking, 0 = unrelated)')

Features in both vocabularies: 38,527 of 50,000
Spearman correlation of their weights: 0.097   (1 = same ranking, 0 = unrelated)


In [17]:
# ISOT 模型最重要的 40 个特征，在 WELFake 模型里的权重和方向
# The ISOT model's 40 strongest features, and their weight / direction in the WELFake model
top = coef_isot.abs().sort_values(ascending=False).index[:40]
comparison = pd.DataFrame({'ISOT weight': coef_isot[top].round(2),
                           'WELFake weight': coef_wel.reindex(top).round(2)})
comparison['same direction'] = np.sign(comparison['ISOT weight']) == np.sign(comparison['WELFake weight'])
print('Same direction:', int(comparison['same direction'].sum()), 'of', len(comparison))
comparison

Same direction: 27 of 40


,ISOT weight,WELFake weight,same direction
said,25.10,14.64,True
said on,18.59,2.55,True
this,-14.28,-5.34,True
us,13.81,-14.63,False
president trump,-13.26,13.80,False
gop,-12.66,4.99,False
just,-12.35,-5.50,True
on wednesday,12.23,-1.20,False
us president,12.20,-3.20,False
on tuesday,12.09,2.43,True


- 两个模型的权重 **几乎不相关**（Spearman 相关系数约 0.1）。ISOT 学到的"什么样的词代表真新闻"，在 WELFake 里基本不成立。
- ISOT 最重要的 40 个特征里，有 **13 个方向完全相反**。最典型的是：
  - **`mr`**：在 ISOT 里是假新闻信号（路透社写 "Trump"，不用 "Mr."），但《纽约时报》的写作规范要求写 "Mr. Trump"，所以在 WELFake 里是 *最强的真新闻信号*（5 万个特征中权重排第 1）。这一个词就会把大量《纽约时报》文章推向"假"。
  - **`president trump`、`gop`、`sen`、`rep`**：在 ISOT 里偏假，在 WELFake 里偏真。
  - **`nov`、`on wednesday`、`us president`**：路透社特有的日期和称谓写法，在 WELFake 里不再代表真新闻。
- **仍然成立的** 是一些通用的"新闻 vs 博客"口吻：`said`、`said on`、星期几（真）；`this`、`just`、`you`、`watch`、`hillary`（假）。这正是 E1 的 AUC 还能保持在 0.71 的原因。

The weights are **nearly uncorrelated** (Spearman ≈ 0.1) and **13 of the top 40 ISOT features flip direction**. `mr` is the clearest case: a fake cue in ISOT (Reuters writes "Trump") but the single strongest *real* cue in WELFake (NYT house style is "Mr. Trump"). What still transfers is a generic news-vs-blog voice (`said`, weekdays vs `this`, `just`, `you`), which is why AUC stays at 0.71.

## 16. WELFake has shortcuts too / WELFake 同样有捷径

E2 的 95% 能不能代表"这个任务的真实难度"？用和 `04_leakage_audit` 相同的探针检查 WELFake（train → val）。

Is E2's 95% the task's true difficulty? The same shortcut probes as `04_leakage_audit`, on WELFake (train → val).

In [18]:
probes.run_probe_suite(wel_train, wel_val)

,accuracy %,AUC
完整模型输入 / full model_text,94.5,0.988
只看标题 / title only,78.3,0.854
只看正文前 20 词 / body first 20,88.5,0.955
只看正文后 20 词 / body last 20,77.6,0.850
形状（保留大小写标点）/ shape of content,93.9,0.987
形状（模型输入）/ shape of model_text,71.5,0.793


WELFake 也有自己的捷径：只看正文前 20 个词就有约 88%，只看标点和大小写的"形状"也有约 94%。它的标签同样是按来源网站打的，所以 E2 的 95% 同样包含了"认出来源"的成分。

所以，**E2 不是"这个任务的真实上限"**，只是"在同一批来源内部能做到多好"的参照。E1 和 E2 的差距说明：**无论在哪个数据集上训练，模型主要学到的都是那个数据集里各个来源的写作习惯，换一批来源就不再适用。**

WELFake has shortcuts of its own (first 20 words ≈ 88%, shape ≈ 94%), so E2 is a within-source reference, not the true ceiling. Whatever dataset a model is trained on, it mostly learns that dataset's sources' habits.

## 17. Supplementary: the reverse direction / 补充：反方向

如果反过来，在 WELFake 上训练、在 ISOT 上测试（ISOT val），结果如何？如果两个方向都大幅下降，就说明问题不是 ISOT 独有的，而是"按来源打标签"的数据集的共同问题。

The reverse — train on WELFake, test on ISOT val. If both directions collapse, the problem is shared by source-labelled datasets in general.

In [19]:
prob_reverse = B.predict_proba_real(vec_wel, clf_wel, isot_val)
pd.DataFrame({'WELFake model → ISOT val': B.evaluate(isot_val['label'], prob_reverse)}).T

,n,real %,majority baseline %,accuracy %,balanced accuracy %,AUC,F1 fake,F1 real,predicted real %
WELFake model → ISOT val,6074.0,55.0,55.0,69.2,67.5,0.766,0.591,0.753,69.8


反方向同样大幅下降（见上表），说明 **在按来源打标签的数据集上训练，模型学到的都是来源本身的习惯**，这不是 ISOT 独有的问题。

The reverse direction drops sharply too: training on any source-labelled dataset teaches that dataset's sources.

## 18. Conclusions / 结论

**1. ISOT 上 98.6% 的准确率几乎不能迁移。** 换成来源不同的新闻后，准确率 56.7%，和"全部猜真"的 56.1% 基本相同；balanced accuracy 60.4%，AUC 0.714。同一批测试文章上，在 WELFake 内部训练的模型可以达到 95.0%。

**2. ISOT 模型实际学到的是"像不像路透社"。** 它把 56.8% 的《纽约时报》真新闻判为假新闻。原因可以直接在权重里看到：两个模型的权重几乎不相关，ISOT 最重要的 40 个特征里有 13 个方向相反，比如 `mr`，路透社不写 "Mr."，而《纽约时报》要求写。

**3. 能迁移的只是很少一部分通用信号**："said"、星期几这类新闻报道口吻，以及 "this"、"you"、"just" 这类博客口吻。这让 AUC 保持在 0.71，但远远不够实用。

**4. 这不是 ISOT 独有的问题。** WELFake 同样有自己的来源捷径，反方向的迁移也大幅下降。只要标签是按来源网站打的，模型学到的就主要是来源的写作习惯。

**对整个项目的意义**：
- 原论文以及之前 notebook 中 97–99% 的准确率，衡量的是"识别 ISOT 里的来源"，**不能** 解读为"能识别假新闻"。
- 后面的步骤要围绕 **跨来源泛化** 来评估。BiLSTM 和 DistilBERT 都要同时报告同数据集分数和跨数据集分数。
- 要真正检验"判断真假"，需要逐条核查的标签（LIAR）或基于证据的方法（步骤 5）。

**局限**：WELFake 的标签同样按来源打，Breitbart 被整体标为"真"本身就有争议；这里只用了一个简单的线性模型；WELFake 中还有少量残留痕迹，例如被空格打断的推文链接 "pic. twitter. com"（只出现在 3.2% 的真新闻中，WELFake 模型把 `pic twitter` 当作真新闻信号）。它只会让 E2 参照略微偏高，对 E1 几乎没有影响（ISOT 模型的词表里没有这个词组），所以只会让"不能迁移"的结论更保守。预训练模型（DistilBERT）能否迁移得更好，是第 2 步要回答的问题。

---

1. **ISOT's 98.6% barely transfers**: 56.7% accuracy vs a 56.1% majority baseline (balanced 60.4%, AUC 0.714) on other sources, where a WELFake-trained model reaches 95.0% on the same articles.
2. **The ISOT model learned "Reuters-like or not"**: it calls 56.8% of NYT articles fake; weights are nearly uncorrelated and 13 of the top 40 features flip (e.g. `mr`).
3. **Only generic news-vs-blog voice transfers** (`said`, weekdays vs `this`, `you`), keeping AUC at 0.71.
4. **Not unique to ISOT**: WELFake has its own source shortcuts and the reverse direction also drops.

**Implications**: the 97–99% figures measure source recognition, not fake-news detection; later steps must report cross-source results; truthfulness needs fact-checked labels (LIAR) or evidence-based methods (step 5). **Limitations**: WELFake is source-labelled too (Breitbart = "real" is debatable); a residual artefact remains in WELFake (tweet links broken as "pic. twitter. com", 3.2% of real articles), which can only inflate E2 — "pic twitter" is not in the ISOT vocabulary — making the conclusion conservative; and only a linear model was tested here — whether DistilBERT transfers better is step 2's question.